    # RAG (Retrieval Augmented Generation) with LangChain

Demonstrates a full RAG pipeline:
1. Load documents from CSV
2. Generate embeddings with HuggingFace `all-MiniLM-L6-v2`
3. Store in Chroma vector DB
4. Retrieve relevant docs and answer questions via an LLM

Also shows Pydantic output parsing for structured skill-gap analysis.

In [ ]:
%pip install -q langchain langchain_community langchain_openai langchain_experimental sentence_transformers langchain-chroma python-dotenv pydantic

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY in a .env file"
print("API key loaded.")

## Create sample CSV

Replace with your own `processed.csv` if you have one. The cell below generates a small sample so the notebook runs end-to-end.

In [ ]:
import csv

sample_courses = [
    {"agenda name description": "Python for Data Science", "category": "Python", "level": "Beginner"},
    {"agenda name description": "Advanced Python Flask REST API Development", "category": "Python", "level": "Advanced"},
    {"agenda name description": "AWS Cloud Fundamentals", "category": "Cloud", "level": "Beginner"},
    {"agenda name description": "Microservices with Spring Boot and Kubernetes", "category": "Java", "level": "Advanced"},
    {"agenda name description": "Java 8 Streams and Functional Programming", "category": "Java", "level": "Intermediate"},
    {"agenda name description": "Machine Learning with Python and scikit-learn", "category": "ML", "level": "Intermediate"},
    {"agenda name description": "Docker and Container Orchestration", "category": "Cloud", "level": "Intermediate"},
    {"agenda name description": "React Frontend Development", "category": "Frontend", "level": "Beginner"},
]

with open("processed.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["agenda name description", "category", "level"])
    writer.writeheader()
    writer.writerows(sample_courses)

print("processed.csv created with", len(sample_courses), "rows.")

## Imports

In [ ]:
from langchain_community.document_loaders import CSVLoader
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.globals import set_debug

set_debug(False)

## Load CSV

In [ ]:
loader = CSVLoader("./processed.csv", source_column="agenda name description")
documents = loader.load()
print(f"Loaded {len(documents)} documents")
documents[:1]

## Embeddings + Chroma vector store

In [ ]:
embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
db = Chroma.from_documents(documents, embedding_function)
print("Vector DB ready.")

In [ ]:
query = "Python"
docs = db.similarity_search(query)
print(docs[0].page_content)

## Pydantic output parser — structured skill-gap extraction

In [ ]:
from typing import Optional
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

class SkillGap(BaseModel):
    skill: Optional[list[str]] = Field(default=None, description="Skill gaps identified as short phrases")
    technology: Optional[str]  = Field(default=None, description="Technology area for the skill gap")

class SkillGapList(BaseModel):
    skill_gaps: Optional[list[SkillGap]] = Field(default=None, description="List of identified skill gaps")

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
parser = PydanticOutputParser(pydantic_object=SkillGapList)

prompt = ChatPromptTemplate.from_messages([
    ("system", """
Act as a skill gap analyser. Using the feedback below, identify skill gaps grouped by technology area.
If the input is not feedback reply with NOT_AN_FEEDBACK.
{format_instructions}
"""),
    ("human", "Identify skill gaps for:\n{text}"),
])

runnable = prompt | llm | parser

In [ ]:
feedback = """
Positives:
- Candidate has good Core Java and Java 8 knowledge.
- Familiar with design patterns and Spring Boot.

Needs improvement:
- Microservices and Cloud knowledge.
- Python Flask and REST API development.
"""

result = runnable.invoke({
    "text": feedback,
    "format_instructions": parser.get_format_instructions(),
})

for gap in result.skill_gaps:
    print(f"Technology: {gap.technology}")
    print(f"Skills: {gap.skill}\n")

## RAG chain — recommend courses for each skill gap

In [ ]:
retriever = db.as_retriever()

template = """You are a course recommender. Based on the skill gap below and the available courses, recommend the most relevant course.
If no course matches, reply NO_RECOMMENDATION_AVAILABLE.

Available courses:
{context}

Skill gap: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(template)

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

In [ ]:
for gap in result.skill_gaps:
    query = f"{gap.technology}: {', '.join(gap.skill or [])}"
    recommendation = rag_chain.invoke(query)
    print(f"Gap:  {query}")
    print(f"Rec:  {recommendation}\n")